# Data Preparation — CloudTrail Identity Logs

## Step 1: Load Raw CloudTrail Logs

In [154]:
# !pip install pandas

In [155]:
import pandas as pd
import json
import glob
import os

# path to local CloudTrail folder
cloudtrail_folder = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Raw/CloudTrail/CloudTrail/*.json")

records = []

print("Loading CloudTrail logs...")

for file in glob.glob(cloudtrail_folder):
    with open(file, "r") as f:
        data = json.load(f)
        if "Records" in data:
            records.extend(data["Records"])

print("Total raw records:", len(records))

df_identity = pd.DataFrame(records)

df_identity.describe()

Loading CloudTrail logs...
Total raw records: 2900


,eventVersion,userIdentity,eventTime,eventSource,eventName,awsRegion,sourceIPAddress,userAgent,requestParameters,responseElements,requestID,eventID,readOnly,eventType,managementEvent,recipientAccountId,eventCategory,tlsDetails,resources,errorCode,errorMessage,additionalEventData,sharedEventID,apiVersion,vpcEndpointId,sessionCredentialFromConsole,serviceEventDetails
count,2900,2900,2900,2900,2900,2900,2900,2900,2567,327,2895,2900,2900,2900,2900,2900,2900,2295,693,300,296,310,36,10,155,256,2
unique,2,151,595,29,260,1,16,155,1250,263,2848,2900,2,3,1,1,1,38,79,32,95,290,36,4,5,1,1
top,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:07:57Z,ec2.amazonaws.com,Decrypt,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,"{'accountAttributeNameSet': {}, 'filterSet': {}}","{'version': 1, 'tier': 'Standard'}",95b435ce-68af-4a4b-b89c-f653d8946ebc,5ae76ecb-47d9-4c92-8065-ebe0f8292243,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...","[{'accountId': '123837392027', 'type': 'AWS::K...",ThrottlingException,Rate exceeded,{'ARN': 'arn:aws:secretsmanager:us-east-1:1238...,686ad061-b64c-415b-abe3-5b763d8daff3,20140328,vpce-0d985940f92611b87,true,{'snapshotId': 'snap-0e83092ae538da583'}
freq,2865,2104,110,892,178,2900,2154,768,37,42,3,1,2326,2855,2900,2900,2900,785,164,102,102,2,1,6,76,256,2


In [156]:
# Limit to 1000 rows for prototyping
# df_identity = df_identity.head(1000)

print("Shape:", df_identity.shape)

Shape: (2900, 27)


In [157]:
print("Number of records loaded:", len(records))
print()

if len(records) > 0:
    print("Keys in first record:")
    for key in records[0].keys():
        print(" -", key)
else:
    print("Records list is EMPTY — check your CloudTrail folder path")
    print("Folder being searched:", cloudtrail_folder)
    print()
    import glob, os
    files_found = glob.glob(cloudtrail_folder)
    print("JSON files found:", len(files_found))
    for f in files_found:
        print(" -", f)

Number of records loaded: 2900

Keys in first record:
 - eventVersion
 - userIdentity
 - eventTime
 - eventSource
 - eventName
 - awsRegion
 - sourceIPAddress
 - userAgent
 - requestParameters
 - responseElements
 - requestID
 - eventID
 - readOnly
 - eventType
 - managementEvent
 - recipientAccountId
 - eventCategory
 - tlsDetails


## Step 2: Initial Inspection

In [158]:
df_identity.info()

<class 'pandas.DataFrame'>
RangeIndex: 2900 entries, 0 to 2899
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   eventVersion                  2900 non-null   str   
 1   userIdentity                  2900 non-null   object
 2   eventTime                     2900 non-null   str   
 3   eventSource                   2900 non-null   str   
 4   eventName                     2900 non-null   str   
 5   awsRegion                     2900 non-null   str   
 6   sourceIPAddress               2900 non-null   str   
 7   userAgent                     2900 non-null   str   
 8   requestParameters             2567 non-null   object
 9   responseElements              327 non-null    object
 10  requestID                     2895 non-null   str   
 11  eventID                       2900 non-null   str   
 12  readOnly                      2900 non-null   bool  
 13  eventType                    

In [159]:
df_identity.head()

,eventVersion,userIdentity,eventTime,eventSource,eventName,awsRegion,sourceIPAddress,userAgent,requestParameters,responseElements,requestID,eventID,readOnly,eventType,managementEvent,recipientAccountId,eventCategory,tlsDetails,resources,errorCode,errorMessage,additionalEventData,sharedEventID,apiVersion,vpcEndpointId,sessionCredentialFromConsole,serviceEventDetails
0,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:06:32Z,iam.amazonaws.com,CreateRole,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,"{'path': '/', 'roleName': 'stratus-red-team-ec...",{'role': {'assumeRolePolicyDocument': '%7B%22S...,f0edd1d5-4012-422b-9cb8-8d2abcf77311,5ae76ecb-47d9-4c92-8065-ebe0f8292243,False,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:06:32Z,iam.amazonaws.com,GetRole,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,{'roleName': 'stratus-red-team-ec2lui-role-pcc...,None,34fde66a-cf70-4de7-94c6-461dad3aea30,bfb909d2-000b-4b3c-977f-36ec74c5701c,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:06:32Z,iam.amazonaws.com,ListRolePolicies,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,{'roleName': 'stratus-red-team-ec2lui-role-pcc...,None,cc6ef3ca-3a64-4a96-94b5-4699c548d4d1,f56f75a6-1118-4a10-834d-d54fa7523259,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:06:32Z,iam.amazonaws.com,ListAttachedRolePolicies,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,{'roleName': 'stratus-red-team-ec2lui-role-pcc...,None,ba0f8d9f-b401-4ea1-9b26-7ccbc6a8b4cb,70d264b5-4b90-44b8-8348-2575839c2ca0,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:06:35Z,ec2.amazonaws.com,AssociateRouteTable,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,"{'routeTableId': 'rtb-06711e8f3a624506d', 'sub...",{'requestId': '5bd25912-c738-4545-bb91-e4cfbd8...,5bd25912-c738-4545-bb91-e4cfbd84ad97,08d0a6cd-769b-4e65-89cb-d865f85abbe1,False,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [160]:
df_identity.tail()

,eventVersion,userIdentity,eventTime,eventSource,eventName,awsRegion,sourceIPAddress,userAgent,requestParameters,responseElements,requestID,eventID,readOnly,eventType,managementEvent,recipientAccountId,eventCategory,tlsDetails,resources,errorCode,errorMessage,additionalEventData,sharedEventID,apiVersion,vpcEndpointId,sessionCredentialFromConsole,serviceEventDetails
2895,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:12:10Z,ec2.amazonaws.com,DescribeAvailabilityZones,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,"{'availabilityZoneSet': {}, 'availabilityZoneI...",None,f3f47174-0abf-42b6-a163-1abdbadb92db,9bc58f61-ae58-42b8-8f67-e4b0075571cf,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2896,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:12:14Z,iam.amazonaws.com,GetUser,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,None,None,ac0952d1-ec84-4ea8-9362-8efdc97fb0d5,8d2b37b8-b127-4d06-9f5d-45c2b0e39af1,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2897,1.08,"{'type': 'AWSService', 'invokedBy': 'rolesanyw...",2023-07-10T12:28:25Z,sts.amazonaws.com,AssumeRole,us-east-1,rolesanywhere.amazonaws.com,rolesanywhere.amazonaws.com,{'roleArn': 'arn:aws:iam::123837392027:role/aw...,{'credentials': {'accessKeyId': 'ASIATFQR7NSCY...,2263599c-fea9-4928-a1d7-690a203efa92,2a2e7bf7-30b6-422d-a58f-cef4be3a6239,True,AwsApiCall,True,123837392027,Management,NaN,"[{'accountId': '123837392027', 'type': 'AWS::I...",NaN,NaN,NaN,5030aceb-68ab-41a9-a4c9-21aed6af1e00,NaN,vpce-08931dbe1a3ad50e0,NaN,NaN
2898,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:28:32Z,ec2.amazonaws.com,DescribeVpcAttribute,us-east-1,192.168.10.20,APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...,{'vpcId': 'vpc-07b33857c7ad1c027'},None,30e6c997-7199-4ae7-ba9f-30ba407846bf,24090004-722e-4400-b21a-0c5cb6c8a9eb,True,AwsApiCall,True,123837392027,Management,"{'tlsVersion': 'TLSv1.2', 'cipherSuite': 'ECDH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2899,1.08,"{'type': 'IAMUser', 'principalId': 'AIDATFQR7N...",2023-07-10T12:29:14Z,resource-explorer-2.amazonaws.com,ListIndexes,us-east-1,10.8.8.10,Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:102...,{'Type': 'AGGREGATOR'},None,85e252ed-c0df-4b96-8292-9ee90547f8ca,1da456c0-b61e-4fe4-9213-887e6c6003b4,True,AwsApiCall,True,123837392027,Management,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Note on Missing Values
High missing % in columns like `errorCode`, `responseElements`, `vpcEndpointId`, and `additionalEventData` is **expected and normal** in CloudTrail.
These fields only appear when a specific event condition is triggered (e.g., `errorCode` only exists if the API call failed).
This is **structurally sparse data by design**, not a data quality problem.

## Step 3: Flatten userIdentity (Nested JSON Column)

In [161]:
# userIdentity is a nested dict — must be flattened before any ML use
user_identity_df = pd.json_normalize(df_identity['userIdentity'])

print("Flattened userIdentity columns:")
print(user_identity_df.columns.tolist())

# Merge flattened columns back into main dataframe
df_identity = pd.concat([df_identity.drop(columns=['userIdentity']), user_identity_df], axis=1)

print("\nNew shape after flattening:", df_identity.shape)

Flattened userIdentity columns:
['type', 'principalId', 'arn', 'accountId', 'accessKeyId', 'userName', 'invokedBy', 'sessionContext.attributes.creationDate', 'sessionContext.attributes.mfaAuthenticated', 'sessionContext.sessionIssuer.type', 'sessionContext.sessionIssuer.principalId', 'sessionContext.sessionIssuer.arn', 'sessionContext.sessionIssuer.accountId', 'sessionContext.sessionIssuer.userName', 'sessionContext.ec2RoleDelivery']

New shape after flattening: (2900, 41)


In [162]:
# Set display options to show everything
pd.set_option('display.max_columns', None)

# Now call your describe or head
print(df_identity.describe(include='all'))

       eventVersion             eventTime        eventSource eventName  \
count          2900                  2900               2900      2900   
unique            2                   595                 29       260   
top            1.08  2023-07-10T12:07:57Z  ec2.amazonaws.com   Decrypt   
freq           2865                   110                892       178   

        awsRegion sourceIPAddress  \
count        2900            2900   
unique          1              16   
top     us-east-1   192.168.10.20   
freq         2900            2154   

                                                userAgent  \
count                                                2900   
unique                                                155   
top     APN/1.0 HashiCorp/1.0 Terraform/1.1.2 (+https:...   
freq                                                  768   

                                       requestParameters  \
count                                               2567   
unique           

In [163]:
df_identity.info()

<class 'pandas.DataFrame'>
RangeIndex: 2900 entries, 0 to 2899
Data columns (total 41 columns):
 #   Column                                      Non-Null Count  Dtype 
---  ------                                      --------------  ----- 
 0   eventVersion                                2900 non-null   str   
 1   eventTime                                   2900 non-null   str   
 2   eventSource                                 2900 non-null   str   
 3   eventName                                   2900 non-null   str   
 4   awsRegion                                   2900 non-null   str   
 5   sourceIPAddress                             2900 non-null   str   
 6   userAgent                                   2900 non-null   str   
 7   requestParameters                           2567 non-null   object
 8   responseElements                            327 non-null    object
 9   requestID                                   2895 non-null   str   
 10  eventID                            

In [164]:
# the number of unique names
unique_user_count = df_identity['userName'].nunique()
print(f"Total Unique Users: {unique_user_count}")

print(df_identity['userName'].value_counts())

# the total number of rows
total_rows = len(df_identity)
print(f"Total Event Logs: {total_rows}")

Total Unique Users: 3
userName
bert-jan                              2642
benjamin                               105
stratus-red-team-nmfalu-gfjyeaypjt       1
Name: count, dtype: int64
Total Event Logs: 2900


## Step 4: Handle Missing Values

In [165]:
# Compute missing value report BEFORE any fixes
missing = df_identity.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(df_identity)) * 100
missing_table = pd.concat([missing, missing_percent], axis=1)
missing_table.columns = ["Missing Count", "Missing %"]

# Show only columns that actually have missing values
print("Columns with missing values (before fix):")
missing_table[missing_table["Missing Count"] > 0].head(20)

Columns with missing values (before fix):


,Missing Count,Missing %
serviceEventDetails,2898,99.931034
apiVersion,2890,99.655172
sessionContext.ec2RoleDelivery,2877,99.206897
sharedEventID,2864,98.758621
sessionContext.sessionIssuer.userName,2824,97.379310
sessionContext.sessionIssuer.accountId,2824,97.379310
sessionContext.sessionIssuer.arn,2824,97.379310
sessionContext.sessionIssuer.principalId,2824,97.379310
sessionContext.sessionIssuer.type,2824,97.379310
vpcEndpointId,2745,94.655172


In [166]:
# These are the identity columns needed for the HGNN graph nodes and edges.
# Missing here = ambiguous identity = treat as 'Unknown' (preserves the signal).
identity_cols = [
    'type',
    'principalId',
    'accountId',
    'accessKeyId',
    'userName',
    'invokedBy',
    'sessionContext.attributes.mfaAuthenticated',
    'sessionContext.sessionIssuer.type',
    'sessionContext.sessionIssuer.userName'
]

# Only apply fillna to columns that actually exist (safety check)
existing_identity_cols = [c for c in identity_cols if c in df_identity.columns]
df_identity[existing_identity_cols] = df_identity[existing_identity_cols].fillna("Unknown")

# Verify fix was applied
print("Missing values in identity columns AFTER fix:")
print(df_identity[existing_identity_cols].isnull().sum())

Missing values in identity columns AFTER fix:
type                                          0
principalId                                   0
accountId                                     0
accessKeyId                                   0
userName                                      0
invokedBy                                     0
sessionContext.attributes.mfaAuthenticated    0
sessionContext.sessionIssuer.type             0
sessionContext.sessionIssuer.userName         0
dtype: int64


In [167]:
# For error-related columns, NaN means 'no error occurred' — fill with 'NoError'
error_cols = ['errorCode', 'errorMessage']
for col in error_cols:
    if col in df_identity.columns:
        df_identity[col] = df_identity[col].fillna("NoError")

print("Error columns filled. Sample:")
print(df_identity['errorCode'].value_counts().head())

Error columns filled. Sample:
errorCode
NoError                         2600
ThrottlingException              102
Client.UnauthorizedOperation      44
AccessDenied                      16
NoSuchBucketPolicy                14
Name: count, dtype: int64


## Step 5: Extract Time Features from eventTime

In [168]:
df_identity['eventTime'] = pd.to_datetime(df_identity['eventTime'], utc=True)

df_identity['event_hour']      = df_identity['eventTime'].dt.hour
df_identity['event_dayofweek'] = df_identity['eventTime'].dt.dayofweek
df_identity['event_date']      = df_identity['eventTime'].dt.date
df_identity['is_weekend']      = df_identity['event_dayofweek'].isin([5, 6]).astype(int)

# is_off_hours: access outside 08:00-18:00 = potential anomaly signal
df_identity['is_off_hours'] = (~df_identity['event_hour'].between(8, 18)).astype(int)

print("Time features added.")
df_identity[['eventTime', 'event_hour', 'event_dayofweek', 'is_weekend', 'is_off_hours']].head()

Time features added.


,eventTime,event_hour,event_dayofweek,is_weekend,is_off_hours
0,2023-07-10 12:06:32+00:00,12,0,0,0
1,2023-07-10 12:06:32+00:00,12,0,0,0
2,2023-07-10 12:06:32+00:00,12,0,0,0
3,2023-07-10 12:06:32+00:00,12,0,0,0
4,2023-07-10 12:06:35+00:00,12,0,0,0


In [169]:
df_identity[['eventTime', 'event_hour', 'event_dayofweek', 'is_weekend', 'is_off_hours']].describe()

,event_hour,event_dayofweek,is_weekend,is_off_hours
count,2900.000000,2900.0,2900.0,2900.0
mean,11.724828,0.0,0.0,0.0
std,0.446678,0.0,0.0,0.0
min,11.000000,0.0,0.0,0.0
25%,11.000000,0.0,0.0,0.0
50%,12.000000,0.0,0.0,0.0
75%,12.000000,0.0,0.0,0.0
max,12.000000,0.0,0.0,0.0


## Step 6: Create Anomaly Labels
This is a **rule-based heuristic label** for prototype purposes.
An event is flagged as anomalous if it matches known attacker patterns in the Invictus-IR dataset.

In [170]:
# Source: Invictus-IR attack scenario documentation
HIGH_RISK_EVENTS = [
    'AssumeRole',
    'CreateAccessKey',
    'AttachUserPolicy',
    'AttachRolePolicy',
    'PutUserPolicy',
    'CreateLoginProfile',
    'UpdateLoginProfile',
    'GetSecretValue',
    'ListBuckets',
    'GetObject',
]

# Rule: flag as anomaly if high-risk event AND (off-hours OR no MFA OR there is an errorCode that is not NoError)
is_high_risk_event = df_identity['eventName'].isin(HIGH_RISK_EVENTS)
is_no_mfa = df_identity.get('sessionContext.attributes.mfaAuthenticated', pd.Series(['Unknown']*len(df_identity))) != 'true'
is_failed = (df_identity['errorCode'] != 'NoError') & (df_identity['errorCode'].notnull())

df_identity['is_anomaly'] = (
    is_high_risk_event & (is_no_mfa | is_failed)
).astype(int)

print("Label distribution:")
print(df_identity['is_anomaly'].value_counts())
print(f"\nAnomaly rate: {df_identity['is_anomaly'].mean():.2%}")

Label distribution:
is_anomaly
0    2780
1     120
Name: count, dtype: int64

Anomaly rate: 4.14%


## Step 7: Select Final Columns for HGNN Graph Construction
These columns map directly to the graph schema defined in the thesis methodology:
- **Nodes**: `userName` (identity), `sourceIPAddress` (network bridge), `eventSource` (target service)
- **Edges**: defined by `eventName` (what action was taken)
- **Features**: time features, MFA status, error status
- **Label**: `is_anomaly`

In [171]:
FINAL_COLS = [
    # --- GRAPH NODES (The Skeleton) ---
    'userName',         # Will become 'user' nodes
    'sourceIPAddress',  # Will become 'ip' nodes
    'eventSource',      # Will become 'service' nodes
    
    # --- NODE FEATURES (The Intelligence) ---
    'eventName',        # Crucial: distinguishes specific actions
    'readOnly',         # Distinguishes 'Discovery' from 'Modification'
    'errorCode',        # Critical: the #1 indicator of an attack
    
    # --- IDENTITY CONTEXT ---
    'type',             # Distinguishes IAMUser vs AssumedRole
    'sessionContext.attributes.mfaAuthenticated', # Security posture check
    
    # --- TARGET LABEL ---
    'is_anomaly'        # Your fixed ground truth
]

# Only keep columns that exist in the dataframe
final_cols_existing = [c for c in FINAL_COLS if c in df_identity.columns]
df_final = df_identity[final_cols_existing].copy()

print("Final dataframe shape:", df_final.shape)
print("\nRemaining missing values:")
print(df_final.isnull().sum()[df_final.isnull().sum() > 0])

df_final.head()

Final dataframe shape: (2900, 9)

Remaining missing values:
Series([], dtype: int64)


,userName,sourceIPAddress,eventSource,eventName,readOnly,errorCode,type,sessionContext.attributes.mfaAuthenticated,is_anomaly
0,bert-jan,192.168.10.20,iam.amazonaws.com,CreateRole,False,NoError,IAMUser,Unknown,0
1,bert-jan,192.168.10.20,iam.amazonaws.com,GetRole,True,NoError,IAMUser,Unknown,0
2,bert-jan,192.168.10.20,iam.amazonaws.com,ListRolePolicies,True,NoError,IAMUser,Unknown,0
3,bert-jan,192.168.10.20,iam.amazonaws.com,ListAttachedRolePolicies,True,NoError,IAMUser,Unknown,0
4,bert-jan,192.168.10.20,ec2.amazonaws.com,AssociateRouteTable,False,NoError,IAMUser,Unknown,0


In [174]:
df_final.isnull().sum()

userName                                      0
sourceIPAddress                               0
eventSource                                   0
eventName                                     0
readOnly                                      0
errorCode                                     0
type                                          0
sessionContext.attributes.mfaAuthenticated    0
is_anomaly                                    0
dtype: int64

In [173]:
# Save the cleaned dataframe for the next stage (Graph Construction)
output_path = os.path.expanduser("~/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/df_identity_clean.csv")
df_final.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print("Data preparation complete. Ready for graph construction.")

Saved to: /Users/philberttan/Downloads/federated-hgnn-anomaly-detection/Dataset/Processed/df_identity_clean.csv
Data preparation complete. Ready for graph construction.
